# 66. Constrained combiners, against the collapsing offset

**One variable against ledger row 149**: the combiner. Same 139 members, same folds, same
fold-wise protocol. Only the function mapping member logits to a score changes.

## The problem this is aimed at

The CV-to-leaderboard offset is collapsing as public members arrive, and the collapse is in the
realisation rate rather than in the direction:

| stack | CV gain | LB gain | realised |
|---|---|---|---|
| row 145, +5 public | +0.000337 | +0.00033 | 98% |
| row 146, +10 public | +0.000125 | +0.00009 | 72% |
| row 147, +34 library | +0.000331 | +0.00020 | **60%** |

CV is rising faster than the leaderboard. Two mechanisms are available and they are not exclusive.

**First, member optimism.** Many of these authors early-stop on the very fold they predict, so
their out-of-fold values were chosen knowing those rows' labels. srcK's manifest discloses this
per member. That optimism is present in all five folds equally, so **no fold-wise combiner
protocol can see it**: fitting on four folds and scoring the fifth does not help when the fifth
fold's member values are themselves optimistic.

**Second, and this is what the notebook tests, the combiner exploits it.** Row 149's fitted
coefficients include `cat_nat_c1` at -0.2146, `trees300` at -0.0743 and `cat_raw` at -0.0914.
Large negative weights in a stack are the classic signature of a combiner cancelling noise between
correlated members. An unconstrained logistic regression over 139 correlated vectors has enormous
freedom to do that, and every unit of it that comes from member optimism is a unit that will not
appear on the leaderboard.

## The arms

| arm | combiner | why |
|---|---|---|
| `logistic` | `LogisticRegression(C=1.0)`, row 149's | the incumbent |
| `shrunk` | `LogisticRegression(C=0.01)` | same family, heavy L2, less freedom to cancel |
| `nonneg` | non-negative least squares on logits | removes negative weights entirely |
| `hillclimb` | greedy forward selection with replacement, on fold AUC | what the public 0.9712 notebooks use |
| `rankavg` | plain rank average of the top 20 members by solo AUC | the crudest constrained blend |

Every arm is fitted inside the fold loop, so no arm is scored on a row whose weights it saw.

## The prediction, and it is deliberately not about CV

**The incumbent should win on CV and that proves nothing.** An unconstrained fit has strictly more
freedom than the constrained ones, so it must fit the out-of-fold matrix at least as well. The
question is which arm transfers, and **CV cannot answer it**, because the quantity being inflated
sits inside the CV.

So this notebook's output is a ranking to submit against, not a verdict. **The expectation is
`logistic` first on CV by +0.0002 to +0.0008, and `nonneg` or `hillclimb` within +0.0005 of it.**
If a constrained arm lands close on CV, it is worth a submission slot precisely because its
leaderboard number should be a larger fraction of its CV.

## The case against

If member optimism is small and the real driver of the collapsing offset is that the public pool
simply overlaps our own members, then constraining the weights removes real signal and every
constrained arm loses on both axes. Row 24 measured this repo's combiner as barely regularised,
`C` from 0.01 to 3.0 moving CV by 2e-6, which says the unconstrained fit was not straining **on
our own 25 members**. It says nothing about 139 members of mixed provenance.


In [1]:
# 37_stack_views.ipynb
# Membership gate for the five vectors produced by notebooks 35 and 36.
# Runs locally: every member vector already lives in artifacts/oof.
import hashlib
import json
import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(b for b in [Path.cwd(), *Path.cwd().parents]
            if (b / "data" / "raw" / "train.csv").exists())
O = ROOT / "artifacts" / "oof"
S = ROOT / "submissions"

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy(np.int8)

# The fold vector is rebuilt rather than loaded, and then checked. A silently different
# fold vector is the one error here that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
FOLD_SHA = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
assert FOLD_SHA == "ec282b0968059676", FOLD_SHA
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")
print(f"fold sha {FOLD_SHA}  VERIFIED")

train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]
fold sha ec282b0968059676  VERIFIED


In [2]:
# Row 94's forty-one minus the duplicate, in row 94's order, then the five candidates.
BASE = [
    ("te42", "te_bag42"), ("te2024", "te_seed2024"), ("te7", "te_seed7"),
    ("te2025", "te_seed2025"), ("te13", "te_seed13"),
    ("anchor", "lgbm_default_anchor_seed42"), ("trees300", "lgbm_trees300_seed42"),
    ("trees1000", "lgbm_trees1000_seed42"), ("trees2000", "lgbm_trees2000_seed42"),
    ("lr010", "lgbm_lr01_n1000_seed42"), ("lr005", "lgbm_lr005_n2000_seed42"),
    ("lr003", "lgbm_lr003_n3333_seed42"),
    ("bag42", "lgbm_bag08_lr005_n2000_seed42"),
    ("bag2024", "lgbm_bag08_lr005_n2000_seed2024"),
    ("bag7", "lgbm_bag08_lr005_n2000_seed7"),
    ("bag2025", "lgbm_bag08_lr005_n2000_seed2025"),
    ("bag13", "lgbm_bag08_lr005_n2000_seed13"),
    ("neural", "neural"), ("cat42", "catboost_te"), ("cat2024", "catboost_te_seed2024"),
    ("cat7", "catboost_te_seed7"), ("cat2025", "catboost_te_seed2025"),
    ("cat13", "catboost_te_seed13"), ("neural_te", "neural_te"),
    ("xgb_te", "xgb_te"), ("xgb2024", "xgb_te_seed2024"), ("xgb7", "xgb_te_seed7"),
    ("xgb2025", "xgb_te_seed2025"), ("xgb13", "xgb_te_seed13"),
    ("pair_top9", "xgb_pair_top9"),
]
BASE += [("cat_nat_c1", "cat_native_c1"), ("cat_nat_c2", "cat_native_c2"),
         ("xgb_raw", "xgb_raw"), ("cat_raw", "cat_raw"),
         ("xgb_te_fe", "xgb_te_fe"), ("cat_te_n4000", "cat_te_n4000"),
         ("xgb_raw_fe", "xgb_raw_fe"), ("cat_raw_n10k", "cat_raw_n10000"),
         ("lgb_raw_fe", "lgb_raw_fe"), ("cat_raw_fe", "cat_raw_fe")]
# lgb_raw is deliberately absent: it is bag42 re-run in another kernel, Pearson 0.999970 on
# logits. See the header and the 2026-08-22 entry in NOTES.md. Dropping it costs -0.000001.
DROPPED = [("lgb_raw", "duplicate of bag42, Pearson 0.999970")]

BASE += [("cat_te_fe", "cat_te_fe"), ("hgb_te_fe", "hgb_te_fe"),
         ("lgb_te_fe", "lgb_te_fe"), ("rf_te_fe", "rf_te_fe"),
         ("neural_lookup", "neural_lookup"), ("et_te_fe", "et_te_fe"),
         ("logit_te_fe", "logit_te_fe")]
BASE += [("xgb_tuned", "xgb_tuned")]
BASE += [("neural_fe", "neural_fe"), ("neural_wide", "neural_wide"),
         ("neural_res", "neural_res")]
BASE += [("realmlp", "realmlp")]
BASE += [("realmlp10", "realmlp10")]
BASE += [("realmlp_raw_fe", "realmlp_raw_fe")]
BASE += [("tabm", "tabm")]
# THE ONE VARIABLE. Row 144 held these fifty-five, all built by this repo. The five
# candidates below were not. Each is admitted only by writeup/verify_public_oof.py,
# which proves the fold partition matches ours by reproducing the author's own printed
# per-fold AUCs from our fold vector. See the header.
# Row 145's five, now part of the base.
PRIOR_PUBLIC = ["cb_srcE", "lgb_srcE", "xgb_srcA", "realmlp_srcA", "tabm_srcA",
                "lgb_srcB", "realmlp_srcI", "hgb_srcH", "xgb_srcC", "resnet_kava"]
PUBLIC = PRIOR_PUBLIC
# THE ONE VARIABLE. Thirty-four members from the frozen-fold library, loaded as raw
# .npy rather than through the kernel gate, because their provenance is established by
# transitivity against pub_rmlp and pub_tabm being bit-identical. See the header.
PRIOR_SZY = sorted((ROOT / "artifacts" / "srcL" / "picked.txt").read_text().split())
_szy_raw = sorted((ROOT / "artifacts" / "srcL" / "picked2.txt").read_text().split())
PRIOR_SZY2 = [f"szy_{n}" for n in _szy_raw]
# srcK members are single letters, so they are namespaced like the srcL batch.
srcK = [f"gol_{m}" for m in "abcdefg"]
# This batch contains `realmlp` and `xgb_tuned`, which collide with OUR member aliases.
# Namespaced so the loader cannot silently overwrite one of ours.
SZY = []
_SZY_FILE = {f"szy_{n}": n for n in _szy_raw}
# No new members. Row 149 holds all of these; the combiner is the variable.
CAND = [(n, n) for n in srcK]


def load(stem, kind):
    # OOF/test vector. Two naming conventions exist in artifacts/oof. The bare
    # "{stem}.npy" form is the OOF side only: the early LightGBM members never had a
    # test .npy written and their test side lives in submissions/. Falling back to the
    # bare name for kind="test" silently returns the OOF vector, which is caught by the
    # length assert below only because train and test differ in length.
    cands = [O / f"{stem}_{kind}.npy"]
    if kind == "oof":
        cands.append(O / f"{stem}.npy")
    for c in cands:
        if c.exists():
            return np.load(c)
    if kind == "test" and (S / f"{stem}.csv").exists():
        df = pd.read_csv(S / f"{stem}.csv")
        # A csv written in a different row order blends perfectly cleanly and is
        # undetectable in the score. Checked rather than assumed.
        assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {stem}"
        return df["addicted_label"].to_numpy()
    raise FileNotFoundError(f"{stem} {kind}")


# Row 145 already holds PRIOR_PUBLIC, so they belong in the index alongside our own.
# Row 147 holds BASE + PRIOR_PUBLIC + PRIOR_SZY, so all three belong in the index.
MEM = (BASE + [(n, n) for n in PRIOR_PUBLIC]
       + [(n, n) for n in PRIOR_SZY + PRIOR_SZY2] + CAND)
names = [n for n, _ in MEM]
Poof = {n: load(s, "oof") for n, s in BASE}
Ptest = {n: load(s, "test") for n, s in BASE}

# The public five, read from artifacts/public_oof/ with their id order asserted rather
# than assumed. A csv in a different row order blends perfectly cleanly and is invisible
# in the score, which is the failure row 59's loader already guards against.
PUB = ROOT / "artifacts" / "public_oof"
VER = {r["name"]: r for r in json.loads((PUB / "verification.json").read_text(encoding="utf-8"))}


def read_vec(path, n_expected, order_ref):
    """Mirror of writeup/verify_public_oof.load_vector, so a member is loaded here
    exactly as it was loaded when it was verified. Two shapes exist in the wild: a
    csv with or without an id column, and a bare .npy."""
    path = pathlib.Path(path)
    if path.suffix == ".npy":
        v = np.load(path)
        assert len(v) == n_expected, f"{path.name} has {len(v)} rows"
        return v.astype(float)
    df = pd.read_csv(path)
    assert len(df) == n_expected, f"{path.name} has {len(df)} rows"
    idc = [c for c in df.columns if c.lower() == "id"]
    if idc:
        # A csv in a different row order blends perfectly cleanly and is invisible in
        # the score, so this is asserted rather than hoped for.
        assert (df[idc[0]].to_numpy() == order_ref).all(), f"id order {path.name}"
    pref = [c for c in df.columns if any(k in c.lower() for k in ("oof", "pred", "prob"))]
    col = pref or [c for c in df.columns
                   if c.lower() not in ("id", "addicted_label", "target", "fold")]
    col = col or [c for c in df.columns if c.lower() != "id"]
    return df[col[0]].to_numpy(float)


for n in PUBLIC:
    r = VER.get(n)
    assert r and r["verdict"].startswith("ADMISSIBLE"), \
        f"{n} is not admissible: {r['verdict'] if r else 'absent'}. Re-run the gate."
    d = PUB / r["folder"]
    Poof[n] = read_vec(d / r["oof_file"], len(train), train["id"].to_numpy())
    Ptest[n] = read_vec(d / r["test_file"], len(test), test["id"].to_numpy())
# The library members. The manifest AUC is asserted, so a truncated or wrong file
# cannot enter quietly.
import csv as _csv
SZD = ROOT / "artifacts" / "srcL"
_man = {r["model"]: float(r["oof_auc"])
        for r in _csv.DictReader((SZD / "manifest.csv").open(encoding="utf-8"))}
for n in PRIOR_SZY + PRIOR_SZY2 + SZY:
    stem = _SZY_FILE.get(n, n)
    o = np.load(SZD / f"oof_{stem}.npy")
    t = np.load(SZD / f"test_{stem}.npy")
    assert o.shape == (len(train),) and t.shape == (len(test),), n
    _a = roc_auc_score(y, o)
    assert abs(_a - _man[stem]) < 5e-5, f"{n}: AUC {_a:.6f} vs manifest {_man[stem]}"
    Poof[n], Ptest[n] = o.astype(float), t.astype(float)
print(f"{len(PRIOR_SZY) + len(PRIOR_SZY2)} srcL members carried, all matching manifest AUC")

# The srcK library. Its manifest publishes per-fold AUCs, so the ordinary gate applies:
# every member is PROVEN on our partition rather than accepted on its README.
import csv as _csv2
GOL = ROOT / "artifacts" / "srcK"
for _r in _csv2.DictReader((GOL / "manifest.csv").open(encoding="utf-8")):
    _m = _r["member"]
    _o = np.load(GOL / f"oof_{_m}.npy")
    _t = np.load(GOL / f"test_{_m}.npy")
    assert _o.shape == (len(train),) and _t.shape == (len(test),), _m
    _ours = [roc_auc_score(y[folds == f], _o[folds == f]) for f in range(5)]
    _theirs = [float(_r[f"fold{f}_auc"]) for f in range(5)]
    _d = max(abs(a - b) for a, b in zip(_ours, _theirs))
    assert _d < 1e-4, f"srcK {_m}: fold AUCs differ by {_d:.2e}, a different partition"
    Poof[f"gol_{_m}"], Ptest[f"gol_{_m}"] = _o.astype(float), _t.astype(float)
print(f"{len(srcK)} srcK members verified against their published per-fold AUCs")
print(f"{len(PUBLIC)} kernel-verified public members loaded")
print(f"  kernel-verified     : {len(PRIOR_PUBLIC)}")
print(f"  library candidates  : {len(SZY)}")
_rej = [k for k, v in VER.items() if not v["verdict"].startswith("ADMISSIBLE")]
print(f"  rejected by the gate: {_rej}")

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    assert np.isfinite(Poof[n]).all() and np.isfinite(Ptest[n]).all(), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

# Exact-duplicate quarantine. A duplicated array silently DOUBLES that model's weight.
# Row 59 found xgb_pair_base bit-identical to xgb_te this way and excluded it.
seen = {}
for n in names:
    h = hashlib.md5(np.ascontiguousarray(Poof[n]).tobytes()).hexdigest()
    assert h not in seen, f"{n} is bit-identical to {seen[h]}"
    seen[h] = n
print(f"{len(names)} vectors loaded, no exact duplicates")

# THE CORRELATION SCREEN. Hash equality cannot see one configuration run in two kernels:
# the arrays differ by thread-level numerical noise. bag42 and lgb_raw sat in row 94 at
# Pearson 0.999970 and no check fired. Three lines, and it would have caught them.
Lz = np.column_stack([np.clip(np.log(np.clip(Poof[n], 1e-9, 1 - 1e-9)
                                     / (1 - np.clip(Poof[n], 1e-9, 1 - 1e-9))), -30, 30)
                      for n in names])
Cm = np.corrcoef(((Lz - Lz.mean(0)) / Lz.std(0)).T)
np.fill_diagonal(Cm, 0.0)
near = [(names[i], names[j], Cm[i, j])
        for i in range(len(names)) for j in range(i + 1, len(names))
        if abs(Cm[i, j]) > 0.9999]
print(f"pairs above 0.9999: {len(near)}")
for a, b, r in near:
    print(f"  NEAR-DUPLICATE {a} and {b} at {r:+.6f}")
assert not near, "a near-duplicate pair is present, justify it or drop one"
print(f"removed from row 94's set: {[d[0] for d in DROPPED]}")
hi = sorted(((abs(Cm[i, j]), names[i], names[j])
             for i in range(len(names)) for j in range(i + 1, len(names))),
            reverse=True)[:3]
print("most collinear surviving pairs: "
      + ", ".join(f"{a}/{b} {r:.5f}" for r, a, b in hi))

67 srcL members carried, all matching manifest AUC


7 srcK members verified against their published per-fold AUCs
10 kernel-verified public members loaded
  kernel-verified     : 10
  library candidates  : 0
  rejected by the gate: ['lookup_srcA', 'spline_srcD']


139 vectors loaded, no exact duplicates


pairs above 0.9999: 0
removed from row 94's set: ['lgb_raw']
most collinear surviving pairs: realmlp/realmlp10 0.99948, rmlp_lat/rmlp_lat3 0.99942, cat42/cat7 0.99907


In [3]:
def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
IDX = {n: i for i, n in enumerate(names)}
BASE132 = ([IDX[n] for n, _ in BASE] + [IDX[n] for n in PRIOR_PUBLIC]
           + [IDX[n] for n in PRIOR_SZY + PRIOR_SZY2])

print("candidate solo CV, and disagreement with the members it most resembles:")
print(f"  {'candidate':12} {'solo CV':>10} {'rho vs xgb_te':>15} {'rho vs cat42':>14}")
for n, _ in CAND:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    r1 = pd.Series(Poof[n]).corr(pd.Series(Poof["xgb_te"]), method="spearman")
    r2 = pd.Series(Poof[n]).corr(pd.Series(Poof["cat42"]), method="spearman")
    print(f"  {n:12} {cv:10.6f} {r1:15.6f} {r2:14.6f}")

# For scale: how decorrelated are two members that everyone agrees are near-copies?
r_seed = pd.Series(Poof["xgb_te"]).corr(pd.Series(Poof["xgb2024"]), method="spearman")
print(f"\n  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): {r_seed:.6f}")
print("  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.")
print("  This table is context, not a prediction.")

candidate solo CV, and disagreement with the members it most resembles:
  candidate       solo CV   rho vs xgb_te   rho vs cat42


  gol_a          0.964795        0.983312       0.976638


  gol_b          0.940829        0.947454       0.937029


  gol_c          0.934392        0.934901       0.940292


  gol_d          0.962598        0.980328       0.979152


  gol_e          0.964867        0.984568       0.977577


  gol_f          0.964049        0.981889       0.976059


  gol_g          0.942360        0.928579       0.925859



  for scale, xgb_te vs xgb_te_seed2024 (same model, different seed): 0.997344
  NOTES.md refuted reasoning from Spearman to blend value on 66 pairs at r=+0.143.
  This table is context, not a prediction.


In [4]:
from scipy.optimize import nnls

FLOOR_MEAN, FLOOR_FOLDS = 0.00005, 4
COLS = BASE132 + [IDX[n] for n, _ in CAND]
print(f"{len(COLS)} members, identical across every arm\n")


def fit_logistic(Ltr, ytr, Lva, Lte, C_):
    m = LogisticRegression(C=C_, max_iter=4000).fit(Ltr, ytr)
    assert int(np.max(m.n_iter_)) < 4000, "combiner did not converge"
    return m.decision_function(Lva), m.decision_function(Lte), m.coef_[0]


def fit_nonneg(Ltr, ytr, Lva, Lte, _=None):
    """Non-negative least squares on the logit matrix. No member may take a negative
    weight, so the combiner cannot cancel one member against another."""
    w, _r = nnls(Ltr, ytr.astype(float))
    return Lva @ w, Lte @ w, w


def fit_hillclimb(Ltr, ytr, Lva, Lte, n_iter=40, sub=120_000, seed=0):
    """Greedy forward selection WITH replacement, the public ensembling recipe. Start
    empty, repeatedly add whichever member most improves AUC of the running mean.
    Weights are implicit counts, so they are non-negative and sum to one.

    The search scores candidates on a fixed random SUBSAMPLE of the training part.
    Scoring 40 rounds x 139 members on all 553,095 rows is about forty minutes per fold
    and the first attempt at this notebook was killed by its own timeout doing exactly
    that. The subsample only ranks candidates; the returned weights are applied to the
    full validation and test matrices."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(Ltr), size=min(sub, len(Ltr)), replace=False)
    Ls, ys = Ltr[idx], ytr[idx]
    cur = np.zeros(len(Ls))
    picks = []
    for _ in range(n_iter):
        k = len(picks)
        cands = (cur[:, None] * k + Ls) / (k + 1)
        aucs = [roc_auc_score(ys, cands[:, j]) for j in range(Ls.shape[1])]
        j = int(np.argmax(aucs))
        cur = cands[:, j]
        picks.append(j)
    w = np.bincount(picks, minlength=Ltr.shape[1]).astype(float) / len(picks)
    return Lva @ w, Lte @ w, w


def fit_rankavg(Ltr, ytr, Lva, Lte, k=20):
    """Plain rank average of the k strongest members, measured on the training part."""
    aucs = [roc_auc_score(ytr, Ltr[:, j]) for j in range(Ltr.shape[1])]
    top = np.argsort(aucs)[::-1][:k]
    w = np.zeros(Ltr.shape[1])
    w[top] = 1.0 / k
    rv = np.mean([pd.Series(Lva[:, j]).rank(pct=True).to_numpy() for j in top], axis=0)
    rt = np.mean([pd.Series(Lte[:, j]).rank(pct=True).to_numpy() for j in top], axis=0)
    return rv, rt, w


ARMS_C = {
    "logistic": (fit_logistic, 1.0),
    "shrunk": (fit_logistic, 0.01),
    "nonneg": (fit_nonneg, None),
    "hillclimb": (fit_hillclimb, 40),
    "rankavg": (fit_rankavg, 20),
}

res_c, test_c = {}, {}
for arm, (fn, param) in ARMS_C.items():
    t0 = time.time() if "time" in dir() else None
    oof_a = np.zeros(len(train))
    tst_a = np.zeros((5, len(test)))
    for f in range(5):
        tr, va = folds != f, folds == f
        v, t, _w = fn(Loof[np.ix_(tr, COLS)], y[tr],
                      Loof[np.ix_(va, COLS)], Ltest[:, COLS], param)
        oof_a[va] = v
        tst_a[f] = t
    per_a = np.array([roc_auc_score(y[folds == f], oof_a[folds == f]) for f in range(5)])
    res_c[arm] = per_a
    test_c[arm] = tst_a
    print(f"  {arm:10} CV {per_a.mean():.6f} +/- {per_a.std():.6f}")


139 members, identical across every arm



  logistic   CV 0.969755 +/- 0.000392


  shrunk     CV 0.969759 +/- 0.000392


  nonneg     CV 0.967746 +/- 0.000458


  hillclimb  CV 0.969474 +/- 0.000400


  rankavg    CV 0.969174 +/- 0.000404


In [5]:
base = res_c["logistic"]
ROW149_CV = 0.969778
d0 = base.mean() - ROW149_CV
print(f"logistic reproduces row 149: {base.mean():.6f} vs {ROW149_CV:.6f}, delta {d0:+.2e}")
print(f"{'REPRODUCED' if abs(d0) < 1e-4 else 'FAILED, do not log this run'}\n")

print(f"{'arm':10} {'CV':>10} {'vs logistic':>13} {'folds':>7} {'t(4)':>8}")
for arm, per_a in res_c.items():
    d = per_a - base
    sd = d.std(ddof=1)
    t = d.mean() / (sd / np.sqrt(5)) if sd > 0 else 0.0
    print(f"{arm:10} {per_a.mean():10.6f} {d.mean():+13.6f} "
          f"{int((d > 0).sum()):5d}/5 {t:8.2f}")

print("\nCV RANKS THESE AND CANNOT DECIDE THEM. The unconstrained fit has strictly more")
print("freedom, so it must fit the out-of-fold matrix at least as well. The question is")
print("which arm's number survives contact with the test set, and the quantity suspected")
print("of being inflated lives inside CV. The leaderboard is the instrument here.")

# The diagnostic that motivated the notebook: how much negative weight each arm uses.
print("\nnegative weight used by each arm, fold 0, which is the overfitting signature")
for arm, (fn, param) in ARMS_C.items():
    tr, va = folds != 0, folds == 0
    _v, _t, w = fn(Loof[np.ix_(tr, COLS)], y[tr], Loof[np.ix_(va, COLS)],
                   Ltest[:, COLS], param)
    w = np.asarray(w, dtype=float)
    print(f"  {arm:10} sum|w| {np.abs(w).sum():8.4f}   negative mass {np.abs(w[w < 0]).sum():8.4f}"
          f"   members used {int((np.abs(w) > 1e-9).sum()):4d}")

SUB_DIR = ROOT / "submissions"
for arm in ("nonneg", "hillclimb"):
    p = test_c[arm].mean(axis=0)
    sub = pd.DataFrame({"id": test["id"].to_numpy(),
                        "addicted_label": (np.argsort(np.argsort(p)) + 0.5) / len(p)})
    assert len(sub) == len(test) and np.isfinite(sub["addicted_label"]).all()
    sub.to_csv(SUB_DIR / f"stack_{arm}.csv", index=False)
    print(f"\nwrote stack_{arm}.csv, CV {res_c[arm].mean():.6f}")
print("\nTwo constrained arms written for submission. This notebook does not choose")
print("between them on CV, because CV is the thing under suspicion.")


logistic reproduces row 149: 0.969755 vs 0.969778, delta -2.30e-05
REPRODUCED

arm                CV   vs logistic   folds     t(4)
logistic     0.969755     +0.000000     0/5     0.00
shrunk       0.969759     +0.000004     5/5     8.76
nonneg       0.967746     -0.002009     0/5   -55.84
hillclimb    0.969474     -0.000281     0/5   -18.22
rankavg      0.969174     -0.000581     0/5   -15.82

CV RANKS THESE AND CANNOT DECIDE THEM. The unconstrained fit has strictly more
freedom, so it must fit the out-of-fold matrix at least as well. The question is
which arm's number survives contact with the test set, and the quantity suspected
of being inflated lives inside CV. The leaderboard is the instrument here.

negative weight used by each arm, fold 0, which is the overfitting signature


  logistic   sum|w|   5.1974   negative mass   2.1307   members used  139


  shrunk     sum|w|   4.8884   negative mass   1.9784   members used  139


  nonneg     sum|w|   0.2065   negative mass   0.0000   members used    7


  hillclimb  sum|w|   1.0000   negative mass   0.0000   members used   11


  rankavg    sum|w|   1.0000   negative mass   0.0000   members used   20



wrote stack_nonneg.csv, CV 0.967746



wrote stack_hillclimb.csv, CV 0.969474

Two constrained arms written for submission. This notebook does not choose
between them on CV, because CV is the thing under suspicion.


In [6]:
# The coefficient table belongs to the membership notebooks. This one varies the
# combiner, and its diagnostic is the negative-weight table above.
print("see the negative weight table in the previous cell")


see the negative weight table in the previous cell


In [7]:
print("Submissions are written above. Membership is unchanged from row 149.")


Submissions are written above. Membership is unchanged from row 149.
